In [1]:
from torch.utils.data import Dataset
from datasets import load_dataset
from scripts.image_process import process_image, image_to_tensor
from scripts.tokenizer import get_tokenizer
from scripts.ground_truth_process import json2token, process_ground_truth, get_train_inputs
from torch.utils.data import DataLoader

In [2]:
dataset_name = "ridwanFatur98/synth-invoice"
train_data = load_dataset(dataset_name, split="train")
test_data = load_dataset(dataset_name, split="test")

In [3]:
class DonutDataset(Dataset):
    def __init__(
        self, 
        dataset,
        tokenizer,
        
        # Image
        input_size = [1280, 960],
        align_long_axis = False,
        random_padding = True,

        # Ground Truth
        task_start_token = "<s_cord-v2>",
        prompt_end_token = "<s_cord-v2>",
        max_length = 768,
    ):
        super().__init__()
        self.dataset = dataset
        self.tokenizer = tokenizer

        # Image
        self.input_size = input_size
        self.align_long_axis = align_long_axis
        self.random_padding = random_padding

        # Ground Truth
        self.task_start_token = task_start_token
        self.max_length = max_length
        self.prompt_end_token = prompt_end_token

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int):
        sample = self.dataset[idx]

        # Image
        img = process_image(sample["image"], self.input_size, self.align_long_axis, self.random_padding)
        image_tensor = image_to_tensor(img)
        
        # Ground Truth
        ground_truth = sample["ground_truth"]
        gt_token_sequence, _ = process_ground_truth(self.tokenizer, ground_truth, task_start_token=self.task_start_token)
        prompt_end_token_id = tokenizer.convert_tokens_to_ids(self.prompt_end_token)
        input_ids, labels = get_train_inputs(self.tokenizer, gt_token_sequence, prompt_end_token_id, max_length=self.max_length)

        return image_tensor, input_ids, labels

In [4]:
tokenizer = get_tokenizer()

In [5]:
train_dataset = DonutDataset(train_data, tokenizer)

In [6]:
image_tensor, input_ids, labels = train_dataset[0]

In [7]:
image_tensor.shape, input_ids.shape, labels.shape

(torch.Size([3, 1280, 960]), torch.Size([768]), torch.Size([768]))

In [8]:
batch_size = 5

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
)

In [10]:
image_tensors, input_ids, labels = next(iter(train_loader))

In [11]:
image_tensors.shape, input_ids.shape, labels.shape

(torch.Size([5, 3, 1280, 960]), torch.Size([5, 768]), torch.Size([5, 768]))

In [13]:
labels.reshape(-1).shape

torch.Size([3840])

### Model

In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [13]:
import torch.nn as nn
import torch
from scripts.model import DonutConfig, DonutModel

# Encoder Config
encoder_layer = [1, 1, 1, 1]
window_size = 10
input_size = [1280, 960]

# Decoder Config
max_position_embeddings = 768
decoder_layer = 1

# Full Config
config = DonutConfig(
    input_size=input_size,
    window_size=window_size,
    encoder_layer=encoder_layer,
    max_position_embeddings=max_position_embeddings,
    decoder_layer=decoder_layer,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
)

W1001 07:29:28.389000 9700 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1001 07:29:28.421000 9700 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [14]:
model = DonutModel(config).to(device)

In [23]:
image_tensors = image_tensors.to(device)
input_ids = input_ids.to(device)
labels = labels.to(device)

In [24]:
decoder_input_ids = input_ids[:, :-1]
decoder_labels = labels[:, 1:]

decoder_input_ids.shape, decoder_labels.shape

(torch.Size([5, 767]), torch.Size([5, 767]))

In [25]:
decoder_outputs = model(
    image_tensors=image_tensors,
    decoder_input_ids=decoder_input_ids,
    decoder_labels=decoder_labels,
)

[W1001 07:30:47.761427059 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 50331648 bytes (free: 0, total: 4294639616).
[W1001 07:30:47.821150535 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 50331648 bytes (free: 0, total: 4294639616).


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.73 GiB is allocated by PyTorch, and 102.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [18]:
decoder_outputs['loss']

tensor(16.4041, device='cuda:0', grad_fn=<NllLossBackward0>)